# 🚀 AdaptiveSLM - TPU v5e Training

Optimized for Kaggle TPU v5e-8 (fastest training option)

---

In [ ]:
# TPU Setup
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

!pip install -q cloud-tpu-client
!pip install -q torch~=2.1.0 torch_xla~=2.1.0 -f https://storage.googleapis.com/libtpu-releases/index.html
!pip install -q transformers datasets sentencepiece

In [ ]:
import torch
import torch_xla
import torch_xla.core.xla_model as xm
import torch_xla.distributed.parallel_loader as pl
import torch_xla.distributed.xla_multiprocessing as xmp

DEVICE = xm.xla_device()
print(f"TPU Device: {DEVICE}")
print(f"TPU Cores: {xm.xrt_world_size()}")

In [ ]:
# Configuration for TPU
from dataclasses import dataclass

@dataclass
class TPUConfig:
    # Model
    vocab_size: int = 32000
    hidden_size: int = 576
    num_layers: int = 30
    num_attention_heads: int = 9
    num_key_value_heads: int = 3
    intermediate_size: int = 1536
    max_position_embeddings: int = 2048
    
    # Training (TPU-optimized: larger batch)
    batch_size: int = 32  # Per core
    gradient_accumulation_steps: int = 4
    learning_rate: float = 5e-4  # Higher LR for TPU
    max_steps: int = 10000
    
    output_dir: str = "/kaggle/working/checkpoints"

config = TPUConfig()
print(f"Effective batch size: {config.batch_size * config.gradient_accumulation_steps * xm.xrt_world_size()}")

In [ ]:
# Model definition (same as GPU version)
import torch.nn as nn
import torch.nn.functional as F
import math

class RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.eps = eps
    
    def forward(self, x):
        variance = x.pow(2).mean(-1, keepdim=True)
        return self.weight * x * torch.rsqrt(variance + self.eps)

class GroupedQueryAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = self.hidden_size // self.num_heads
        self.num_key_value_groups = self.num_heads // self.num_kv_heads
        
        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)
    
    def forward(self, hidden_states, attention_mask=None):
        batch_size, seq_len, _ = hidden_states.size()
        
        q = self.q_proj(hidden_states).view(batch_size, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(hidden_states).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(hidden_states).view(batch_size, seq_len, self.num_kv_heads, self.head_dim).transpose(1, 2)
        
        k = k.repeat_interleave(self.num_key_value_groups, dim=1)
        v = v.repeat_interleave(self.num_key_value_groups, dim=1)
        
        attn_weights = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        if attention_mask is not None:
            attn_weights = attn_weights + attention_mask
        attn_weights = F.softmax(attn_weights, dim=-1)
        attn_output = torch.matmul(attn_weights, v)
        
        return self.o_proj(attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, -1))

class SwiGLU(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.gate_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, config.intermediate_size, bias=False)
        self.down_proj = nn.Linear(config.intermediate_size, config.hidden_size, bias=False)
    
    def forward(self, x):
        return self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x))

class TransformerBlock(nn.Module):
    def __init__(self, config, layer_idx):
        super().__init__()
        self.input_layernorm = RMSNorm(config.hidden_size)
        self.self_attn = GroupedQueryAttention(config)
        self.post_attention_layernorm = RMSNorm(config.hidden_size)
        self.mlp = SwiGLU(config)
    
    def forward(self, hidden_states, attention_mask=None):
        residual = hidden_states
        hidden_states = self.self_attn(self.input_layernorm(hidden_states), attention_mask)
        hidden_states = residual + hidden_states
        
        residual = hidden_states
        hidden_states = self.mlp(self.post_attention_layernorm(hidden_states))
        return residual + hidden_states

class AdaptiveSLM(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.embed_tokens = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([TransformerBlock(config, i) for i in range(config.num_layers)])
        self.norm = RMSNorm(config.hidden_size)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.lm_head.weight = self.embed_tokens.weight
    
    def forward(self, input_ids, labels=None):
        batch_size, seq_len = input_ids.shape
        hidden_states = self.embed_tokens(input_ids)
        
        causal_mask = torch.triu(torch.full((seq_len, seq_len), float("-inf"), device=input_ids.device), diagonal=1)
        causal_mask = causal_mask.unsqueeze(0).unsqueeze(0)
        
        for layer in self.layers:
            hidden_states = layer(hidden_states, causal_mask)
        
        logits = self.lm_head(self.norm(hidden_states))
        
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[:, :-1].reshape(-1, config.vocab_size), labels[:, 1:].reshape(-1), ignore_index=-100)
        
        return loss, logits

model = AdaptiveSLM(config).to(DEVICE)
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# TPU Training Loop
from datasets import load_dataset
from transformers import AutoTokenizer
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B")
tokenizer.pad_token = tokenizer.eos_token

# Simple dataset
class TextDataset(Dataset):
    def __init__(self, texts, tokenizer, max_length=1024):
        self.encodings = [tokenizer(t, max_length=max_length, truncation=True, padding="max_length", return_tensors="pt") for t in texts]
    
    def __len__(self): return len(self.encodings)
    def __getitem__(self, idx):
        return {"input_ids": self.encodings[idx]["input_ids"].squeeze(), "labels": self.encodings[idx]["input_ids"].squeeze()}

# Load sample data
dataset = load_dataset("HuggingFaceFW/fineweb-edu", "sample-10BT", split="train", streaming=True)
texts = [item["text"][:2000] for item in list(dataset.take(10000))]

train_dataset = TextDataset(texts, tokenizer)
train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True)

# TPU parallel loader
para_loader = pl.ParallelLoader(train_loader, [DEVICE]).per_device_loader(DEVICE)

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate)

# Training
model.train()
for step, batch in enumerate(tqdm(para_loader, total=config.max_steps)):
    input_ids = batch["input_ids"]
    labels = batch["labels"]
    
    loss, _ = model(input_ids, labels)
    loss.backward()
    
    if (step + 1) % config.gradient_accumulation_steps == 0:
        xm.optimizer_step(optimizer)
        optimizer.zero_grad()
    
    if step % 100 == 0:
        xm.master_print(f"Step {step}, Loss: {loss.item():.4f}")
    
    if step >= config.max_steps:
        break

# Save
xm.save(model.state_dict(), f"{config.output_dir}/adaptive_slm_tpu.pt")
print("Training complete!")